# 0.3 · Pandas Deep Dive 全面解析

> **课程定位 / Where this fits**
> 第 3 课，**Part 0 · 基础准备**。
> Lesson 3, **Part 0 · Foundations**.
>
> 0.2 节我们打通了 **NumPy**（同构数值数组）；这一课讲 **Pandas**——把 NumPy 包装成"自带列名 + 缺失值处理 + I/O + groupby + merge"的数据表，**数据分析师/科学家 80% 的日常工作**都在这里。
> Lesson 0.2 covered NumPy (homogeneous numeric arrays). Pandas wraps NumPy into a labelled "spreadsheet on steroids" — column names, missing data, I/O, groupby, joins. **Analysts/scientists live in pandas 80% of the time.**

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> 本节几乎都是数据操作，数学公式很少。涉及统计时遵循：$n$ = 样本数 / row count，$d$ = 列数 / column count，$\bar{x}$ = 样本均值 / sample mean，$s$ = 样本标准差 / sample std。
> Mostly data manipulation; minimal math. Stats follow: $n$ rows, $d$ columns, $\bar{x}$ sample mean, $s$ sample std.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 在 `Series` 和 `DataFrame` 上熟练做**索引、过滤、聚合、连接、重塑**。
   Index, filter, aggregate, join, reshape on `Series` / `DataFrame`.
2. 区分 `.loc[]` / `.iloc[]` / `.at[]` / `.iat[]` 的使用场景。
   Tell apart `.loc[]` / `.iloc[]` / `.at[]` / `.iat[]`.
3. 处理缺失值的三种方式：删除 / 填充 / 标记。
   Handle missing data via drop / fill / flag.
4. 用 `groupby` + `agg`/`transform`/`filter` 完成绝大多数 SQL 风格分析。
   Replicate most SQL analytics with `groupby` + `agg/transform/filter`.
5. 用 `merge` / `pivot_table` / `melt` 重塑数据。
   Reshape with `merge`, `pivot_table`, `melt`.
6. 在 **Titanic** 数据集上完成一份**完整的 EDA**，能讲出"哪种乘客生还率最高、为什么"。
   Deliver a full EDA on Titanic and explain who survived & why.

---

## 目录 / Table of Contents

1. [为什么需要 Pandas / Why Pandas](#1)
2. [🛳 Titanic 数据集介绍 / Dataset Intro](#2)
3. [Series 详解 / Series](#3)
4. [DataFrame 详解 / DataFrame](#4)
5. [I/O 速览 / I/O Overview](#5)
6. [快速查看数据 / Quick Inspection](#6)
7. [索引：`.loc` / `.iloc` / `.at` / `.iat`](#7)
8. [布尔过滤 / Boolean Filtering](#8)
9. [增删改列 / Add, Drop, Rename Columns](#9)
10. [排序 / Sorting](#10)
11. [缺失值处理 / Missing Data](#11)
12. [数据类型 / Dtypes（含 `category` / `datetime`）](#12)
13. [字符串与日期方法 / `.str` and `.dt`](#13)
14. [`apply` / `map` / `replace`](#14)
15. [`groupby` —— Pandas 的核心](#15)
16. [`agg` / `transform` / `filter`](#16)
17. [合并：`merge` / `join` / `concat`](#17)
18. [重塑：`pivot_table` / `melt` / `stack` / `unstack`](#18)
19. [MultiIndex 多级索引](#19)
20. [方法链 + `pipe` / Chaining](#20)
21. [性能小贴士 / Performance](#21)
22. [实战：Titanic 完整 EDA / End-to-end EDA](#22)
23. [小结 / Summary](#23)


<a id="1"></a>
## 1. 为什么需要 Pandas / Why Pandas

NumPy 适合**同质数值**，但真实数据长这样：

| name | age | sex | fare | embarked |
|---|---|---|---|---|
| Allen | 35 | M | 8.05 | S |
| Allison| NaN | F | 53.10 | S |

- 列**类型不同**（字符串 + 数值 + 类别）
- 有**缺失值**
- 需要按**列名**（不是位置）操作
- 来源是 **CSV / Excel / SQL / Parquet**

这些都是 Pandas 的舒适区。
Pandas owns these problems.

Pandas 两大核心：
- **`Series`**：带索引的一维数组（**列**）
- **`DataFrame`**：表格 = 共享行索引的多个 Series（**多列**）


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"seaborn : {sns.__version__}")

# 让 pandas 输出更紧凑 / Compact display
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


<a id="2"></a>
## 2. 🛳 Titanic 数据集介绍 / Dataset Intro

> **来源 / Source**: Kaggle 2012 入门竞赛 *"Titanic: Machine Learning from Disaster"*；seaborn 直接内置一个 891 行的版本。
> Kaggle's classic intro challenge; seaborn ships an 891-row copy.
>
> **背景 / Background**: 1912 年泰坦尼克号沉没，2224 名乘客和船员中 1502 人遇难。数据集记录了**乘客的人口学特征**和**是否生还**。
> The Titanic sank in 1912 with 1502 of 2224 onboard lost. The dataset records demographics and survival.
>
> | 列 / Column | 含义 / Meaning | 类型 / Type |
> |---|---|---|
> | `survived`     | 是否生还 0/1 / survived | int |
> | `pclass`       | 船舱等级 1/2/3 / passenger class | int |
> | `sex`          | 性别 / sex | category |
> | `age`          | 年龄 / age | float (含 NaN) |
> | `sibsp`        | 同船兄弟姐妹+配偶数 / #siblings + spouses | int |
> | `parch`        | 同船父母+孩子数 / #parents + children | int |
> | `fare`         | 船票价格 / fare | float |
> | `embarked`     | 登船港口 C/Q/S / port of embarkation | category (含 NaN) |
> | `class`        | First/Second/Third（`pclass` 的字符串版） | category |
> | `who`          | man / woman / child | category |
> | `adult_male`   | 是否成年男性 | bool |
> | `deck`         | 甲板 A–G（**大量缺失**）/ deck letter (lots of NaN) | category |
> | `embark_town`  | 登船城市全名 / full town name | category |
> | `alive`        | yes/no（`survived` 的字符串版）| category |
> | `alone`        | 是否独自登船 / travelling alone | bool |
>
> **任务 / Task**: 二分类预测 `survived`（**但本节是 EDA，不建模**）。
> Binary classification on `survived` (but this lesson is EDA, no model).
>
> **为什么经典 / Why classic**: 小、多类型、有缺失值、清晰的人物背景故事 ——"数据分析的 Hello World"。
> Small, mixed types, missing values, vivid backstory — the "Hello World" of data analysis.


In [ ]:
titanic = sns.load_dataset("titanic")
print(f"shape : {titanic.shape}")     # (n, d)
print(f"dtypes:")
print(titanic.dtypes)


In [ ]:
titanic.head(3)        # 默认显示前 5 行 / default first 5


<a id="3"></a>
## 3. Series 详解 / Series

`Series` = **带索引的一维数组**。想象成 NumPy 数组 + 一列"行名"。
A `Series` is a 1-D array **plus an index** — like NumPy + row labels.

公式角度看，Series 就是从 index 到 value 的函数：
Formally: $s : I \to V$, mapping from index $I$ to value $V$.


In [ ]:
# 从 list 创建 / From list
s = pd.Series([10, 20, 30, 40], name="my_series")
print(s)

print("\nvalues  :", s.values)
print("index   :", list(s.index))
print("dtype   :", s.dtype)


In [ ]:
# 自定义索引 / Custom index — 这是 pandas 区别于 numpy 的关键
prices = pd.Series(
    [3.5, 1.2, 8.0, 5.0],
    index=["apple", "banana", "cherry", "date"],
    name="price",
)
print(prices)

# 按标签访问 / Label-based access
print("\nprices['cherry'] =", prices["cherry"])

# 按位置访问 / Position-based access
print("prices.iloc[0]   =", prices.iloc[0])

# 向量化运算（继承自 NumPy）/ Vectorization inherited from NumPy
print("\nprices * 1.1:\n", prices * 1.1)


**Series 的关键技巧**：**两个 Series 做算术时按索引对齐**，不是按位置。
**Key trick**: arithmetic on two Series **aligns by index**, not position.


In [ ]:
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])   # 注意顺序和缺项 / different order, missing key

print("a:", a.values, "index:", list(a.index))
print("b:", b.values, "index:", list(b.index))
print("\na + b (auto-align):\n", a + b)


**结果解读 / Read the result**：
- `y`、`z` 两个 key 在两边都有 → 算出 22、33
- `x`（只在 `a` 里）、`w`（只在 `b` 里）→ 结果是 **NaN**

这就是 pandas 引入缺失值的**根本原因之一**：算术对齐时缺项怎么办？答案：NaN。
This index-alignment behavior is one reason pandas needs `NaN`.


<a id="4"></a>
## 4. DataFrame 详解 / DataFrame

`DataFrame` = **多个共享行索引的 Series**。每一列是一个 Series。
A `DataFrame` is multiple Series sharing one row index. Each column is a Series.

实际等价于关系数据库的"表"或 R 的 `data.frame`，但 API 更 Python 化。
Functionally a SQL table or R `data.frame`, with a Pythonic API.


In [ ]:
# 从字典创建（key 变列名）/ From dict (keys become columns)
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Dave"],
    "age":  [25, 30, 35, 40],
    "city": ["NY", "LA", "NY", "SF"],
})
print(df)

print("\nshape :", df.shape)         # (n, d)
print("index :", list(df.index))
print("cols  :", list(df.columns))


In [ ]:
# 选列 / Selecting columns
print("--- 选单列（得到 Series）/ single col (→ Series)---")
print(df["name"])
print("\ntype:", type(df["name"]))

print("\n--- 选多列（得到 DataFrame）/ multi col (→ DataFrame)---")
print(df[["name", "age"]])


> **⚠ 单方括号 vs 双方括号 / Single vs double brackets**
> - `df["col"]` → **Series** （1D）
> - `df[["col"]]` → **DataFrame** （2D，只有一列）
> 这个区别在写 sklearn 时常踩坑：sklearn 多数 API 要求 2D 输入。
> sklearn often requires 2D inputs — this distinction matters.


<a id="5"></a>
## 5. I/O 速览 / I/O Overview

Pandas 的 I/O 库覆盖了大多数数据格式。**实际工作中你 90% 用 `read_csv`**。
Pandas's I/O covers most formats. **90% of real work uses `read_csv`.**

| 函数 / Function | 用途 / Use |
|---|---|
| `pd.read_csv(path, ...)` | CSV |
| `pd.read_excel(path, sheet_name=...)` | Excel |
| `pd.read_json(path)` | JSON |
| `pd.read_parquet(path)` | Parquet（大数据首选 / preferred for big data） |
| `pd.read_sql(query, conn)` | SQL |
| `pd.read_html(url)` | 抓网页表格 / web tables |

写出对称：`df.to_csv(...)` / `df.to_parquet(...)` 等。
Writing is symmetric.


In [ ]:
# 演示：把 titanic 存成 CSV 再读回来 / Round-trip via CSV
from pathlib import Path
tmp = Path("/tmp/titanic_demo.csv")
titanic.to_csv(tmp, index=False)
print(f"wrote {tmp.stat().st_size/1024:.1f} KB")

df = pd.read_csv(tmp)
print(f"reload shape: {df.shape}")


**`read_csv` 最常用的参数 / Most-used `read_csv` args**

```python
pd.read_csv(
    path,
    sep=",",                # 分隔符 / separator (e.g. "\t" for TSV)
    header=0,               # 标题行 / header row
    names=[...],            # 自己指定列名 / override column names
    usecols=[...],          # 只读指定列 / only load some columns
    dtype={"col": "int32"}, # 指定类型节省内存 / cast dtypes
    parse_dates=["date"],   # 自动解析日期 / parse dates
    na_values=["NA", "?"],  # 额外的缺失值标记 / extra NaN markers
    nrows=1000,             # 只读前 N 行 / first N rows
    chunksize=10_000,       # 流式读取（大文件）/ streaming for huge files
)
```

> **大文件处理 / Big files**：CSV 几 GB 时用 `chunksize` 流式 + `concat`，或者直接换 **Parquet**（列存、压缩、读 10× 快）。
> For multi-GB files use `chunksize` or switch to **Parquet** (columnar, compressed, ~10× faster).


<a id="6"></a>
## 6. 快速查看数据 / Quick Inspection

**每次拿到新数据，先跑这五行**：
First five things you do on a new dataset:


In [ ]:
print("--- 1) shape ---")
print(titanic.shape)         # (rows, cols)

print("\n--- 2) head ---")
print(titanic.head(3))       # 前 3 行 / first 3 rows

print("\n--- 3) info ---")
titanic.info()               # dtype + 非空个数 / dtypes + non-null counts


In [ ]:
print("--- 4) describe (数值列统计) ---")
print(titanic.describe())


In [ ]:
print("--- 5) describe (类别列) ---")
print(titanic.describe(include="object"))     # 字符串列 / string cols

print("\n--- 缺失值汇总 / missing-value summary ---")
print(titanic.isna().sum().sort_values(ascending=False).head())


**观察 / Observations** (Titanic):
- `deck` 缺 688 / 891，**77% 缺失** → 多半要丢
- `age` 缺 177 / 891，**20% 缺失** → 必须填充
- `embarked` 缺 2 / 891 → 用众数填就行


<a id="7"></a>
## 7. 索引：`.loc` / `.iloc` / `.at` / `.iat`

**最容易混淆的概念，必须搞清楚。**
**The most confusing concept — you must master it.**

| API | 用什么 / By | 速度 / Speed | 用于 |
|---|---|---|---|
| `df.loc[row_label, col_label]` | **标签** / labels | 普通 | 行/列**切片**、布尔过滤 |
| `df.iloc[row_pos, col_pos]` | **位置** / integer positions | 普通 | 像 NumPy 一样按位置取 |
| `df.at[row_label, col_label]` | 标签 | **最快** | 单格读写 |
| `df.iat[row_pos, col_pos]` | 位置 | **最快** | 单格读写（位置） |

> **关键陷阱 / Key gotcha**
> `df.loc[3]` 是按**索引标签 3** 找，**不**是第 3 行。
> 如果 `index` 默认是 0,1,2,...，两者碰巧一样；但只要 reset 过 index、merge 过、按某列 set_index 过，就完全不一样了。
> `df.loc[3]` looks up label `3`, **not** the 3rd row. If you ever set/reset the index, label ≠ position.


In [ ]:
# .iloc: 像 NumPy 一样按位置
print("titanic.iloc[0]:\n", titanic.iloc[0])           # 第 0 行 / row 0
print("\ntitanic.iloc[0:3, 0:4]:\n", titanic.iloc[0:3, 0:4])


In [ ]:
# .loc: 按标签 / by label
# 当前 index 是 0..890 整数 → loc[0] 和 iloc[0] 一样
print("titanic.loc[0, 'sex']:", titanic.loc[0, "sex"])

# 切片包含末端（和 iloc 不同！）/ loc slice is INCLUSIVE
print("\ntitanic.loc[0:2, ['sex', 'age', 'fare']]:")
print(titanic.loc[0:2, ["sex", "age", "fare"]])


**`.loc` 切片包含右端点，`.iloc` 不包含！**
**`.loc` slice is inclusive of the end; `.iloc` is not!**

```
titanic.loc[0:2]   →  rows with label 0, 1, 2     (3 行 / 3 rows)
titanic.iloc[0:2]  →  rows at position 0, 1       (2 行 / 2 rows)
```

这是为了和 Python 列表 (`[0:2]` 不含 2) **保持一致** vs **像 SQL `BETWEEN`** 一样的两种习惯。


In [ ]:
# .at / .iat: 单格读写最快 / fastest single-cell access
print("at  :", titanic.at[0, "fare"])
print("iat :", titanic.iat[0, 6])


<a id="8"></a>
## 8. 布尔过滤 / Boolean Filtering

**数据分析里写得最多的一种代码。**
**You'll write more of this than anything else.**


In [ ]:
# 单条件 / Single condition
women = titanic[titanic["sex"] == "female"]
print(f"# women: {len(women)}")

# 多条件：用 & | ~，括号必须！/ Combine with & | ~, parens REQUIRED
first_class_kids = titanic[
    (titanic["pclass"] == 1) & (titanic["age"] < 18)
]
print(f"# first-class kids: {len(first_class_kids)}")

# 不在某集合里 / Not in a set — use ~
non_S_ports = titanic[~titanic["embarked"].isin(["S"])]
print(f"# non-S embark: {len(non_S_ports)}")


**陷阱：忘记括号 / Pitfall: missing parens**

```python
# ❌ 错的：& 优先级高于 ==
titanic[titanic["pclass"] == 1 & titanic["age"] < 18]
# Python 把它当成 1 & titanic["age"] —— 完全不是你要的

# ✅ 对的
titanic[(titanic["pclass"] == 1) & (titanic["age"] < 18)]
```


In [ ]:
# query 方法：字符串语法，可读性更好 / String-based filtering
result = titanic.query("pclass == 1 and age < 18")
print(f"first-class kids (via query): {len(result)}")

# 引用外部变量用 @ / Reference external variable with @
threshold = 18
result = titanic.query("pclass == 1 and age < @threshold")
print(f"same with @: {len(result)}")


<a id="9"></a>
## 9. 增删改列 / Add, Drop, Rename Columns


In [ ]:
df = titanic.copy()

# 加列：直接赋值 / Add a column by assignment
df["family_size"] = df["sibsp"] + df["parch"] + 1
print(df[["sibsp", "parch", "family_size"]].head())

# assign：链式风格 / chain-friendly
df2 = (
    titanic
    .assign(family_size=lambda d: d["sibsp"] + d["parch"] + 1,
            fare_per_person=lambda d: d["fare"] / (d["sibsp"] + d["parch"] + 1))
    [["fare", "family_size", "fare_per_person"]]
)
print("\nwith assign:")
print(df2.head())


In [ ]:
# 删列 / Drop columns
df3 = titanic.drop(columns=["deck", "embark_town", "alive"])
print("after drop:", df3.columns.tolist())

# 改名 / Rename
df4 = titanic.rename(columns={"sibsp": "n_siblings", "parch": "n_parents"})
print("after rename:", df4.columns.tolist()[:6])


<a id="10"></a>
## 10. 排序 / Sorting


In [ ]:
# 按某列排序 / Sort by a column
print("top 5 by fare:")
print(titanic.sort_values("fare", ascending=False)[["sex", "pclass", "fare"]].head())

# 多列排序 / Multi-column sort
print("\nby pclass asc, then fare desc:")
sorted_df = titanic.sort_values(["pclass", "fare"], ascending=[True, False])
print(sorted_df[["pclass", "fare", "sex"]].head(8))


<a id="11"></a>
## 11. 缺失值处理 / Missing Data

数据科学家**至少 30% 的时间**花在这上面。
DS spend **at least 30% of their time** here.

| 函数 / Function | 作用 / Purpose |
|---|---|
| `.isna()` / `.isnull()` | 标记缺失 / mark NaN |
| `.notna()` | 反向 / negation |
| `.dropna(axis, how, thresh, subset)` | 删 / drop |
| `.fillna(value)` | 填 / fill |
| `.ffill()` / `.bfill()` | 前向 / 后向填充 / forward / backward |


In [ ]:
print("缺失值数量 / missing counts:")
print(titanic.isna().sum().sort_values(ascending=False).head())

print("\n缺失率 / missing rate:")
print((titanic.isna().mean() * 100).round(1).sort_values(ascending=False).head())


In [ ]:
# 1) 删行：任一列有 NaN 就删 / Drop any row with any NaN
n_before = len(titanic)
n_after = len(titanic.dropna())
print(f"dropna() any: {n_before} -> {n_after}   (太激进 / too aggressive)")

# 2) 删行：subset 指定的列有 NaN 才删
n_after = len(titanic.dropna(subset=["age"]))
print(f"dropna subset=['age']: {n_before} -> {n_after}")

# 3) 删列：某列缺失率 > 50% 才删 / Drop columns with >50% missing
high_miss = titanic.columns[titanic.isna().mean() > 0.5].tolist()
print(f"\ncolumns >50% missing: {high_miss}")


In [ ]:
# 4) 填充：策略对比 / Imputation strategies
df = titanic.copy()
mean_age = df["age"].mean()
median_age = df["age"].median()

df["age_mean_filled"]   = df["age"].fillna(mean_age)
df["age_median_filled"] = df["age"].fillna(median_age)

# 按组填充：每个 pclass 用自己的中位数 / Group-wise median imputation (smarter)
df["age_group_filled"] = df.groupby("pclass")["age"].transform(
    lambda s: s.fillna(s.median())
)

print(df[["age", "age_mean_filled", "age_median_filled", "age_group_filled"]].head(8))


**实战建议 / Practical advice**：
- 数值列 → **中位数**（对异常值稳健）/ median (robust to outliers)
- 类别列 → **众数**或新增一个 "Missing" 类别 / mode or new "Missing" category
- 别盲目 `dropna()`：可能丢掉 90% 数据 / never blindly drop — could lose 90%+
- 按**业务分组**填充几乎总比全局填充好：本例按 `pclass` 分组比整体均值更合理（不同舱位乘客的年龄分布不同）。
  Group-wise imputation usually beats global imputation.


<a id="12"></a>
## 12. 数据类型 / Dtypes

| Pandas dtype | 含义 / Meaning |
|---|---|
| `int64` / `float64` | 数值 / numeric |
| `bool` | 布尔 |
| `object` | Python 对象（**通常是字符串**）/ usually string |
| `string[python]` / `string[pyarrow]` | 显式字符串 / explicit string |
| `category` | **有限取值的类别变量**（节省内存）/ categorical (memory-efficient) |
| `datetime64[ns]` | 时间戳 / timestamp |
| `timedelta64[ns]` | 时间差 / duration |

> 用 `category` 把字符串列转成类别后，**内存可以减少 10× 以上**，groupby 也更快。
> Converting strings to `category` can shrink memory **10×+** and speed up groupby.


In [ ]:
print("dtypes:")
print(titanic.dtypes)

# 内存占用 / Memory usage
print(f"\nmemory before: {titanic.memory_usage(deep=True).sum() / 1024:.1f} KB")

df = titanic.copy()
for col in ["sex", "embarked", "class", "who", "deck", "embark_town", "alive"]:
    df[col] = df[col].astype("category")

print(f"memory after : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"reduction    : {(1 - df.memory_usage(deep=True).sum() / titanic.memory_usage(deep=True).sum()) * 100:.0f}%")


In [ ]:
# 转换数据类型 / Casting
s = pd.Series(["1.5", "2.7", "3.14"])
print(s.astype(float).sum())                # 3.14 + 2.7 + 1.5

# 字符串转日期 / String -> datetime
dates = pd.Series(["2024-01-01", "2024-06-15", "2025-12-31"])
parsed = pd.to_datetime(dates)
print("\nparsed:\n", parsed)
print("dtype :", parsed.dtype)              # datetime64[ns]


<a id="13"></a>
## 13. 字符串与日期方法 / `.str` and `.dt` Accessors

Series 的两类"专属方法"：
Series has two "namespaced" accessors:

- **`.str.<method>`** → 对每个字符串元素操作（vectorized string ops）
- **`.dt.<method>`** → 对每个日期元素操作（year, day_of_week 等）


In [ ]:
# .str 演示 / .str demo
names = pd.Series([
    "Mr. John Smith",
    "Mrs. Mary Jones",
    "Miss. Lisa Brown",
    "Dr.  Bob Lee",
])

print("lower      :", names.str.lower().tolist())
print("contains 'Mr' (含 Miss / Mrs)   :", names.str.contains("Mr").tolist())
print("contains 'Mr.' (精确)           :", names.str.contains(r"^Mr\.").tolist())
print("split first :", names.str.split(" ").str[0].tolist())   # 第一个 token
print("len         :", names.str.len().tolist())
print("title extracted via regex:")
print(names.str.extract(r"^([A-Za-z]+)\."))


In [ ]:
# .dt 演示 / .dt demo
dates = pd.to_datetime([
    "2024-01-15", "2024-06-30", "2025-12-25", "2026-06-06"
])
s = pd.Series(dates)

print("year        :", s.dt.year.tolist())
print("month       :", s.dt.month.tolist())
print("day_of_week :", s.dt.day_of_week.tolist())   # 0=Mon
print("day_name    :", s.dt.day_name().tolist())
print("is_weekend  :", (s.dt.day_of_week >= 5).tolist())


<a id="14"></a>
## 14. `apply` / `map` / `replace`

| 方法 / Method | 作用 / Purpose |
|---|---|
| `Series.map(fn_or_dict)` | 逐元素映射（值替换） |
| `Series.apply(fn)` | 逐元素函数 |
| `DataFrame.apply(fn, axis=0/1)` | 沿轴函数（每列/每行）|
| `Series.replace(old, new)` | 值替换（支持 dict / regex）|

> **重要 / Important**：`apply` 是 Python for-loop 的包装，**比向量化慢 100×+**。能用向量化的绝不用 `apply`。
> `apply` is a Python loop in disguise — **100×+ slower** than vectorized ops. Avoid it when you can.


In [ ]:
# Series.map: 字典映射 / dict mapping
sex_to_int = {"male": 0, "female": 1}
print("first 5:", titanic["sex"].map(sex_to_int).head().tolist())

# Series.apply: 函数 / function
def age_bucket(a):
    if pd.isna(a): return "unknown"
    if a < 18: return "child"
    if a < 60: return "adult"
    return "senior"

buckets = titanic["age"].apply(age_bucket)
print("\nbucket counts:\n", buckets.value_counts())


In [ ]:
# DataFrame.apply axis=1: 行级函数（慎用！慢）
# Row-wise: SLOW — use only when no vectorized alternative
df = titanic.head(5).copy()
df["info"] = df.apply(
    lambda r: f"{r['sex']}-{r['pclass']}-${r['fare']:.0f}",
    axis=1,
)
print(df[["sex", "pclass", "fare", "info"]])


<a id="15"></a>
## 15. `groupby` —— Pandas 的核心 / The Heart of Pandas

**SQL 里的 `GROUP BY` 等价物**，但更强大。任何"按 X 分组计算 Y"的问题都用它。
The equivalent of SQL `GROUP BY` — every "X by Y" question.

**Split–Apply–Combine** 范式：
1. **Split**：按 key 分组
2. **Apply**：对每组施加函数
3. **Combine**：拼回结果


In [ ]:
# 单列分组 + 聚合 / Group by one column + aggregate
print("--- 各舱等的生还率 / survival rate per class ---")
print(titanic.groupby("pclass")["survived"].mean().round(3))

print("\n--- 男女生还率 / survival rate by sex ---")
print(titanic.groupby("sex", observed=True)["survived"].mean().round(3))


In [ ]:
# 多列分组 / Group by multiple columns
print("--- 舱等 × 性别 生还率 / pclass × sex ---")
result = titanic.groupby(["pclass", "sex"], observed=True)["survived"].mean().round(3)
print(result)

# unstack 把内层索引转成列 / pivot the inner index into columns
print("\nunstacked:")
print(result.unstack().round(3))


**观察 / Observation**：
- 一等舱女性生还率 **97%**，三等舱男性生还率仅 **14%**
- "**Women and children first**" 不只是说说的；社会阶层（舱等）也起了巨大作用
- First-class women survived at **97%**, third-class men at **14%** — social class mattered as much as gender.


<a id="16"></a>
## 16. `agg` / `transform` / `filter`

`groupby` 后的三种主要操作：
Three main things you do after `groupby`:

| 方法 / Method | 输入 → 输出 / I → O | 用途 / Use |
|---|---|---|
| `agg(fn)` | 每组 → **一个值** / per-group → scalar | 算汇总：均值、中位数、自定义 |
| `transform(fn)` | 每组 → **同 shape 序列** / same-length output | 在每组内做归一化、填充 |
| `filter(fn)` | 每组 → **保留或删除整组** / keep or drop whole groups | 按组性质过滤 |


In [ ]:
# agg: 多种聚合 / multiple aggregations
agg_result = titanic.groupby("pclass").agg(
    mean_age=("age", "mean"),
    median_fare=("fare", "median"),
    survival_rate=("survived", "mean"),
    n_passengers=("survived", "size"),
).round(3)
print(agg_result)


In [ ]:
# transform: 在每组内计算、保留原 shape
# 例：每个乘客的票价相对于本舱等中位数的比值
df = titanic[["pclass", "fare"]].copy()
df["fare_med_by_class"] = df.groupby("pclass")["fare"].transform("median")
df["fare_ratio"] = df["fare"] / df["fare_med_by_class"]
print(df.head(8))


In [ ]:
# filter: 只保留满足组条件的整组 / keep groups satisfying a condition
# 例：只保留人数 > 100 的舱等组
big = titanic.groupby("pclass").filter(lambda g: len(g) > 100)
print(f"shape after filter: {big.shape}")
print(big["pclass"].value_counts())


<a id="17"></a>
## 17. 合并：`merge` / `join` / `concat`

| 函数 / Function | 作用 / Purpose | 类比 SQL |
|---|---|---|
| `pd.concat([df1, df2])` | 上下/左右拼接 / stack | `UNION ALL` |
| `df1.merge(df2, on=...)` | 按 key 关联 / join on key | `JOIN` |
| `df1.join(df2)` | 按**索引**关联 / join on index | `JOIN ON df1.idx=df2.idx` |


In [ ]:
# 造两张小表演示 merge / Toy tables for merge
employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "Dave"],
    "dept_id": [10, 20, 10, 30],
})
departments = pd.DataFrame({
    "dept_id": [10, 20, 40],
    "dept_name": ["Engineering", "Sales", "HR"],
})

print("employees:\n", employees)
print("\ndepartments:\n", departments)


In [ ]:
# inner join: 两边都有 / both sides
inner = employees.merge(departments, on="dept_id", how="inner")
print("--- inner ---")
print(inner)

# left join: 保留左表所有 / keep all left
left = employees.merge(departments, on="dept_id", how="left")
print("\n--- left ---")
print(left)

# outer join: 保留两边所有 / keep all
outer = employees.merge(departments, on="dept_id", how="outer")
print("\n--- outer ---")
print(outer)


In [ ]:
# concat: 上下拼接 / stack rows
a = pd.DataFrame({"x": [1, 2], "y": [10, 20]})
b = pd.DataFrame({"x": [3, 4], "y": [30, 40]})
print("vertical concat:\n", pd.concat([a, b], ignore_index=True))

# 左右拼接 / stack columns
c = pd.DataFrame({"z": [100, 200]})
print("\nhorizontal concat:\n", pd.concat([a, c], axis=1))


<a id="18"></a>
## 18. 重塑：`pivot_table` / `melt` / `stack` / `unstack`

**长表 ↔ 宽表的相互转换**。数据可视化和报告必备。
**Long ↔ Wide reshaping** — essential for plotting and reporting.


In [ ]:
# pivot_table: 长 → 宽 / long to wide
# 把"舱等 × 性别 → 生还率"做成两维表
pivot = titanic.pivot_table(
    values="survived",
    index="pclass",
    columns="sex",
    aggfunc="mean",
    observed=True,
).round(3)
print("survival rate (pclass × sex):")
print(pivot)


In [ ]:
# melt: 宽 → 长 / wide to long
# 反向操作，把上面的宽表压回长表
long = pivot.reset_index().melt(id_vars="pclass", var_name="sex", value_name="survival_rate")
print("back to long:")
print(long)


In [ ]:
# stack / unstack 是 melt / pivot 的索引层级版
# Stack moves columns into the inner row level; unstack does the reverse.
result = titanic.groupby(["pclass", "sex"], observed=True)["survived"].mean()
print("--- stacked (MultiIndex Series) ---")
print(result)

print("\n--- unstacked (pclass × sex DataFrame) ---")
print(result.unstack())


<a id="19"></a>
## 19. MultiIndex 多级索引

groupby 多个列后自动出现 MultiIndex。它**强大但也是 bug 之源**。
MultiIndex shows up after multi-key groupby — powerful but a frequent source of bugs.


In [ ]:
mi = titanic.groupby(["pclass", "sex"], observed=True).agg(
    mean_age=("age", "mean"),
    survival_rate=("survived", "mean"),
).round(2)
print("MultiIndex DataFrame:\n", mi)

print("\nindex levels :", mi.index.names)

# 按外层索引选 / select by outer index
print("\nmi.loc[1]:\n", mi.loc[1])

# 按完整索引选 / select by full tuple
print("\nmi.loc[(1, 'female')]:")
print(mi.loc[(1, "female")])


In [ ]:
# 拉平回普通列 / Flatten back to regular columns
flat = mi.reset_index()
print(flat)


<a id="20"></a>
## 20. 方法链 + `pipe` / Method Chaining

**Pandas 鼓励"流水线式写法"**：每一步只做一件事，从上往下读像在读 SQL。
**Pandas encourages "pipeline-style"**: one step per line, reads top-to-bottom like SQL.


In [ ]:
# 反例：中间变量满天飞 / Anti-pattern: scattered intermediate variables
df = titanic.copy()
df = df.dropna(subset=["age"])
df["family_size"] = df["sibsp"] + df["parch"] + 1
df = df[df["pclass"].isin([1, 2])]
result_bad = df.groupby("pclass")["survived"].mean()
print("anti-pattern result:\n", result_bad.round(3))


In [ ]:
# 正例：方法链 / Method chain — same result, way cleaner
result = (
    titanic
    .dropna(subset=["age"])
    .assign(family_size=lambda d: d["sibsp"] + d["parch"] + 1)
    .query("pclass in [1, 2]")
    .groupby("pclass")["survived"]
    .mean()
    .round(3)
)
print("chained result:\n", result)


In [ ]:
# pipe: 把任意函数插入链中 / Insert any function into the chain
def add_age_bucket(d):
    return d.assign(
        age_bucket=pd.cut(d["age"], bins=[0, 18, 60, 100],
                          labels=["child", "adult", "senior"])
    )

result = (
    titanic
    .dropna(subset=["age"])
    .pipe(add_age_bucket)
    .groupby("age_bucket", observed=True)["survived"]
    .mean()
    .round(3)
)
print(result)


<a id="21"></a>
## 21. 性能小贴士 / Performance Tips

1. **绝不用 `iterrows()` / `itertuples()` 做计算**。它们比向量化慢 100×。
   Never use `iterrows()` for math. 100× slower than vectorized.
2. **`apply(axis=1)` 是变相循环**，能不用就不用。
   `apply(axis=1)` is also a loop in disguise.
3. **类别列 → `category`**：节省内存 + groupby 提速。
   Convert strings to `category` for memory + speed.
4. **eval / query**：复杂表达式可以用 numexpr 加速。
5. **PyArrow backend**：pandas 2.0+ 支持 `pd.options.mode.dtype_backend = "pyarrow"`，更快更省。
6. 数据量到 GB 级别 → 考虑 **Polars / DuckDB / Spark**（后面章节会讲）。


In [ ]:
import time

# 反例：iterrows / Anti-pattern
df = titanic.copy()

t0 = time.perf_counter()
result_bad = []
for _, row in df.iterrows():
    result_bad.append((row["sibsp"] + row["parch"] + 1))
t_bad = time.perf_counter() - t0

# 正例：向量化 / Vectorized
t0 = time.perf_counter()
result_good = df["sibsp"] + df["parch"] + 1
t_good = time.perf_counter() - t0

print(f"iterrows  : {t_bad*1000:6.1f} ms")
print(f"vectorize : {t_good*1000:6.1f} ms")
print(f"speedup   : ×{t_bad/t_good:.0f}")


<a id="22"></a>
## 22. 实战：Titanic 完整 EDA / End-to-end EDA

用一节内容学到的全部工具做一份 EDA 报告。流程：
A full EDA using everything in this notebook:

1. **加载 + 概览** / load & overview
2. **缺失值处理** / missing values
3. **单变量分布** / univariate
4. **二变量关系** / bivariate vs. `survived`
5. **可视化** / plots
6. **结论** / takeaways


In [ ]:
# Step 1: 加载 / Load
df = sns.load_dataset("titanic").copy()
print(f"shape: {df.shape}")
df.head(3)


In [ ]:
# Step 2: 缺失值 / Missing values
miss = df.isna().mean().sort_values(ascending=False)
print("missing rate:")
print((miss[miss > 0] * 100).round(1).astype(str) + "%")


In [ ]:
# 处理策略 / Strategy
# - deck: 77% 缺失 → 丢弃
# - age: 20% 缺失 → 按 pclass 中位数填
# - embarked / embark_town: < 1% → 用众数填
df_clean = (
    df
    .drop(columns=["deck", "alive"])               # alive 是泄漏 / leakage (alive ≡ survived)
    .assign(
        age=lambda d: d.groupby("pclass")["age"].transform(lambda s: s.fillna(s.median())),
        embarked=lambda d: d["embarked"].fillna(d["embarked"].mode()[0]),
        embark_town=lambda d: d["embark_town"].fillna(d["embark_town"].mode()[0]),
    )
)
print("missing after cleaning:", df_clean.isna().sum().sum())


In [ ]:
# Step 3: 单变量分布 / Univariate
print("--- survived counts ---")
print(df_clean["survived"].value_counts(normalize=True).round(3))

print("\n--- sex ---")
print(df_clean["sex"].value_counts(normalize=True).round(3))

print("\n--- pclass ---")
print(df_clean["pclass"].value_counts(normalize=True).round(3))

print("\n--- age describe ---")
print(df_clean["age"].describe().round(1))

print("\n--- fare describe ---")
print(df_clean["fare"].describe().round(1))


In [ ]:
# Step 4: 二变量 vs survived / Bivariate vs survived
print("--- by sex ---")
print(df_clean.groupby("sex", observed=True)["survived"].mean().round(3))

print("\n--- by pclass ---")
print(df_clean.groupby("pclass")["survived"].mean().round(3))

print("\n--- by sex × pclass ---")
print(df_clean.groupby(["sex", "pclass"], observed=True)["survived"]
      .mean().unstack().round(3))

# 派生特征 / Engineered feature
df_clean = df_clean.assign(
    family_size=lambda d: d["sibsp"] + d["parch"] + 1,
    is_alone=lambda d: (d["sibsp"] + d["parch"]) == 0,
    age_bucket=lambda d: pd.cut(d["age"], bins=[0, 12, 18, 60, 100],
                                labels=["child", "teen", "adult", "senior"]),
    fare_bucket=lambda d: pd.qcut(d["fare"], q=4, labels=["Q1", "Q2", "Q3", "Q4"]),
)
print("\n--- by age_bucket ---")
print(df_clean.groupby("age_bucket", observed=True)["survived"].mean().round(3))

print("\n--- by family_size ---")
print(df_clean.groupby("family_size")["survived"].agg(["mean", "size"]).round(3).head(10))


In [ ]:
# Step 5: 可视化 / Plotting
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sns.barplot(data=df_clean, x="sex", y="survived", ax=axes[0, 0])
axes[0, 0].set_title("Survival by Sex")
axes[0, 0].set_ylabel("survival rate")

sns.barplot(data=df_clean, x="pclass", y="survived", ax=axes[0, 1])
axes[0, 1].set_title("Survival by Class")

sns.barplot(data=df_clean, x="age_bucket", y="survived", ax=axes[0, 2])
axes[0, 2].set_title("Survival by Age Bucket")

sns.barplot(data=df_clean, x="family_size", y="survived", ax=axes[1, 0])
axes[1, 0].set_title("Survival by Family Size")

sns.histplot(data=df_clean, x="age", hue="survived", multiple="stack",
             bins=30, ax=axes[1, 1])
axes[1, 1].set_title("Age distribution by survival")

sns.boxplot(data=df_clean, x="pclass", y="fare", hue="survived", ax=axes[1, 2])
axes[1, 2].set_title("Fare by Class × Survival")
axes[1, 2].set_yscale("log")

plt.tight_layout()
plt.show()


### 📊 EDA 结论 / Takeaways

1. **性别是最强信号** / Sex is the strongest signal —— 女性 **74%** 生还，男性 **19%**。
   Women: 74% survival; men: 19%.
2. **舱等显著影响** / Class matters —— 一等 **63%** vs 三等 **24%**。
   First class 63% vs third 24%.
3. **交互效应** / Interaction —— 一等舱**女性 97%** vs 三等舱**男性 14%**——差了 7 倍。
   First-class women 97% vs third-class men 14% — 7× gap.
4. **儿童略有优势** / Children somewhat favored —— `< 12` 的孩子生还率比成年男性高很多，但比成年女性低。
   "Women and children first" — but the "women" part dominates.
5. **家庭规模呈非线性** / Family size is non-monotonic —— 单身 **30%**，3–4 人家庭 **55%**，但 5+ 人反而降回 **20%** 以下。
   Solo: 30%; family of 3–4: 55%; family ≥5: <20%.
6. **`alive` 是数据泄漏** / `alive` is a leakage column —— 它就是 `survived` 的字符串版，建模时**必须删掉**。
   `alive` is literally `survived` as text — drop before modeling.


<a id="23"></a>
## 23. 小结 / Summary

| 主题 / Topic | 关键 / Key takeaway |
|---|---|
| `Series` / `DataFrame` | 带标签的 NumPy；算术按 index 对齐 |
| I/O | 90% 是 `read_csv`；大数据用 Parquet |
| 索引 | `.loc` 按标签包含右端，`.iloc` 按位置不含 |
| 过滤 | `df[mask]` 或 `df.query(...)`；多条件加括号 |
| 缺失值 | 分组中位数比全局均值更智能 |
| dtype | 字符串列转 `category` 内存减 10× |
| `.str` / `.dt` | 向量化字符串/日期操作 |
| `apply` | 慢，能向量化不用它 |
| `groupby` | split → apply → combine |
| `agg` / `transform` / `filter` | 每组返回值/同 shape/整组保留 |
| `merge` | 默认 inner；多想想要不要 left join |
| `pivot_table` / `melt` | 长 ↔ 宽 |
| MultiIndex | 强大但容易踩坑；考虑 `reset_index()` |
| 方法链 | 像 SQL 一样从上往下读 |
| 性能 | 永远不用 `iterrows`，向量化或换 Polars |

### 工业场景速查 / Cheat sheet

| 任务 / Task | 一行写法 / One-liner |
|---|---|
| 看缺失率 | `df.isna().mean().sort_values(ascending=False)` |
| 按组中位数填 | `df["x"] = df.groupby("g")["x"].transform(lambda s: s.fillna(s.median()))` |
| value_counts + % | `df["col"].value_counts(normalize=True)` |
| 多列聚合 | `df.groupby("g").agg(a=("x","mean"), b=("y","sum"))` |
| 列 z-score | `(df["x"] - df["x"].mean()) / df["x"].std()` |
| 分位数分箱 | `pd.qcut(df["x"], q=4, labels=["Q1","Q2","Q3","Q4"])` |
| 等距分箱 | `pd.cut(df["x"], bins=[0,18,60,100])` |
| 移除高缺失列 | `df.dropna(axis=1, thresh=int(len(df)*0.5))` |
| 时间窗口聚合 | `df.set_index("ts").resample("1D").mean()` |

### 下一节预告 / Next up

**Part 0.4 · Polars 入门** —— 同样是数据表 API，但底层是 Rust + lazy evaluation，**比 pandas 快 5–30 倍**。
**Part 0.4 · Polars Intro** — DataFrame API with a Rust + lazy backend, **5–30× faster** than pandas.
